# Calibrated Linear SVM

## Objective

Add probability estimates to the Linear SVM model so it can support
confidence-based automatic routing and manual review.

This notebook will:

- Recreate the training and test split
- Build a Linear SVM text-classification pipeline
- Calibrate its decision scores using sigmoid calibration
- Evaluate classification performance after calibration
- Evaluate confidence-based routing
- Compare the calibrated SVM with Logistic Regression
- Save the selected pipeline locally

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Locate the project root from either the notebooks folder or project folder.
current_directory = Path.cwd()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

data_path = project_root / "data" / "processed" / "cleaned_tickets.csv"

df = pd.read_csv(data_path)

X = df["ticket_text"]
y = df["queue"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset shape:", df.shape)
print("Training tickets:", len(X_train))
print("Test tickets:", len(X_test))
print("Missing text values:", X.isna().sum())
print("Missing labels:", y.isna().sum())

Dataset shape: (11814, 2)
Training tickets: 9451
Test tickets: 2363
Missing text values: 0
Missing labels: 0


In [2]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

calibration_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

calibrated_svm = CalibratedClassifierCV(
    estimator=svm_pipeline,
    method="sigmoid",
    cv=calibration_cv,
    ensemble=True
)

print("Calibrated SVM configured.")

Calibrated SVM configured.


In [3]:
calibrated_svm.fit(X_train, y_train)

print("Training and calibration complete.")
print("Categories:", calibrated_svm.classes_)
print(
    "Number of calibrated models:",
    len(calibrated_svm.calibrated_classifiers_)
)

Training and calibration complete.
Categories: ['Billing and Payments' 'Customer Service' 'Product Support'
 'Technical Support']
Number of calibrated models: 5


In [4]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score
)

calibrated_predictions = calibrated_svm.predict(X_test)

print(
    f"Test accuracy: "
    f"{accuracy_score(y_test, calibrated_predictions):.2%}"
)

print(
    f"Macro F1: "
    f"{f1_score(y_test, calibrated_predictions, average='macro'):.4f}"
)

print("\nClassification report:")
print(classification_report(
    y_test,
    calibrated_predictions,
    digits=3
))

Test accuracy: 65.51%
Macro F1: 0.6494

Classification report:
                      precision    recall  f1-score   support

Billing and Payments      0.836     0.765     0.799       319
    Customer Service      0.572     0.496     0.531       482
     Product Support      0.611     0.488     0.542       615
   Technical Support      0.658     0.808     0.725       947

            accuracy                          0.655      2363
           macro avg      0.669     0.639     0.649      2363
        weighted avg      0.652     0.655     0.648      2363



In [5]:
calibrated_probabilities = calibrated_svm.predict_proba(X_test)

print("Probability matrix shape:", calibrated_probabilities.shape)

first_ticket_probabilities = pd.Series(
    calibrated_probabilities[0],
    index=calibrated_svm.classes_,
    name="probability"
)

print("\nFirst test ticket probabilities:")
print(first_ticket_probabilities.sort_values(ascending=False))

print("\nActual category:", y_test.iloc[0])
print("Predicted category:", calibrated_predictions[0])
print("Probability sum:", calibrated_probabilities[0].sum())

Probability matrix shape: (2363, 4)

First test ticket probabilities:
Technical Support       0.487995
Customer Service        0.391994
Product Support         0.115471
Billing and Payments    0.004540
Name: probability, dtype: float64

Actual category: Customer Service
Predicted category: Technical Support
Probability sum: 1.0


In [7]:
confidence_results = pd.DataFrame({
    "confidence": calibrated_probabilities.max(axis=1),
    "correct": calibrated_predictions == y_test.to_numpy()
})

confidence_results["confidence_range"] = pd.cut(
    confidence_results["confidence"],
    bins=[0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    include_lowest=True
)

calibration_summary = (
    confidence_results
    .groupby("confidence_range", observed=True)
    .agg(
        ticket_count=("correct", "size"),
        average_confidence=("confidence", "mean"),
        observed_accuracy=("correct", "mean")
    )
)

calibration_summary

,ticket_count,average_confidence,observed_accuracy
confidence_range,,,
"(-0.001, 0.4]",183,0.364238,0.251366
"(0.4, 0.5]",558,0.454407,0.483871
"(0.5, 0.6]",579,0.548837,0.592401
"(0.6, 0.7]",448,0.648294,0.765625
"(0.7, 0.8]",318,0.746091,0.880503
"(0.8, 0.9]",196,0.846595,0.959184
"(0.9, 1.0]",81,0.931828,0.962963


Next, evaluate the same 60% threshold used on Day 6. Keeping it fixed lets us compare routing results without choosing a threshold based on these test results.

In [8]:
confidence_threshold = 0.60

automatic_mask = (
    confidence_results["confidence"] >= confidence_threshold
)

automatic_count = int(automatic_mask.sum())
manual_review_count = len(confidence_results) - automatic_count
automatic_coverage = automatic_mask.mean()

automatic_accuracy = confidence_results.loc[
    automatic_mask, "correct"
].mean()

print("Automatically classified tickets:", automatic_count)
print("Manual-review tickets:", manual_review_count)
print(f"Automatic coverage: {automatic_coverage:.2%}")
print(f"Accuracy on automatic tickets: {automatic_accuracy:.2%}")

Automatically classified tickets: 1043
Manual-review tickets: 1320
Automatic coverage: 44.14%
Accuracy on automatic tickets: 85.23%


## Calibrated SVM Findings

- Test accuracy: 65.51%.
- Macro F1: 0.6494.
- Higher confidence was associated with higher observed accuracy.
- Probability estimates were not perfectly calibrated: the lowest-confidence
  group was overconfident, while higher-confidence groups were generally
  underconfident.

At the fixed 60% confidence threshold:

- 1,043 tickets qualified for automatic routing.
- 1,320 tickets required manual review.
- Automatic coverage was 44.14%.
- Accuracy among automatically routed tickets was 85.23%.

Compared with balanced Logistic Regression, this automated 416 more tickets,
but automatic-routing accuracy decreased by 1.37 percentage points.

This is an exploratory comparison on the previously inspected test set.
Any future threshold tuning should use validation data within the training
set, with routing costs considered before deployment.

In [9]:
import joblib

model_directory = project_root / "models"
model_directory.mkdir(parents=True, exist_ok=True)

calibrated_model_path = (
    model_directory / "calibrated_svm_pipeline.joblib"
)

joblib.dump(calibrated_svm, calibrated_model_path)

print("Saved to:", calibrated_model_path)
print("File exists:", calibrated_model_path.exists())

Saved to: /Users/yogithagujjarlapudi/AI-ML Course/AI-ML-PROJECTS/AI-ML/Customer-Support-Ticket-Intelligence/models/calibrated_svm_pipeline.joblib
File exists: True


In [10]:
import numpy as np

loaded_calibrated_svm = joblib.load(calibrated_model_path)

reloaded_predictions = loaded_calibrated_svm.predict(X_test)
reloaded_probabilities = loaded_calibrated_svm.predict_proba(X_test)

print(
    "Predictions match:",
    np.array_equal(calibrated_predictions, reloaded_predictions)
)

print(
    "Probabilities match:",
    np.allclose(calibrated_probabilities, reloaded_probabilities)
)

Predictions match: True
Probabilities match: True
